# Network Coverage Maps


The following datasets are Network Coverage Maps data for 6 days following the 24 June 2026 earthquake in Venezuela. Network Coverage Maps show where people on Facebook have cellular connectivity. Each row is a location where coverage was detected that day, both National and Caracas level.

In [1]:
import zipfile
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd


def find_project_root(marker: str = "pyproject.toml") -> Path:
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
NETWORK_DIR = DATA_DIR / "Network Coverage Maps"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "ven_admin_boundaries.geojson"

DATASETS = {
    "National": "venezuela_earthquake_network_coverage_june.zip",
    "Caracas": "caracus_earthquake_venezuela.zip",
}
DATASET_COLORS = {"National": "#1F6FB2", "Caracas": "#D6604D"}
BASE_TILES = "CartoDB positron"

In [2]:
HDX_BOUNDARIES_URL = (
    "https://data.humdata.org/dataset/5b141d29-534f-4f01-a0bc-41e2f375d925/"
    "resource/a76o36e17-418e-461b-aa4d-f613f7c8dda5/download/ven_admin_boundaries.geojson.zip"
)
ADMIN_LAYER = "ven_admin1.geojson"

ven_adm1 = gpd.read_file(BOUNDARIES_DIR / ADMIN_LAYER)

In [3]:
def build_persistence(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Per unique tile, how many of the 6 days it appeared (1-6)."""
    return (
        df.assign(day=df["date"].dt.date)
        .groupby(["lon", "lat"], as_index=False)["day"]
        .nunique()
        .rename(columns={"day": "days_present"})
    )


def load_coverage(zip_path: Path) -> pd.DataFrame:
    """Read every daily CSV inside a coverage zip into one long DataFrame."""
    frames = []
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith(".csv"))
        for name in names:
            date = name.split("_")[1]
            with zf.open(name) as fh:
                df = pd.read_csv(fh)
            df["date"] = pd.to_datetime(date)
            frames.append(df)
    return pd.concat(frames, ignore_index=True)


coverage = {name: load_coverage(NETWORK_DIR / fn) for name, fn in DATASETS.items()}

national_coverage = (
    coverage["National"]
    .pipe(build_persistence)
    .assign(geometry=lambda df: gpd.points_from_xy(df["lon"], df["lat"]))
    .pipe(gpd.GeoDataFrame, crs="EPSG:4326")
)
caracas_coverage = (
    coverage["Caracas"]
    .pipe(build_persistence)
    .assign(geometry=lambda df: gpd.points_from_xy(df["lon"], df["lat"]))
    .pipe(gpd.GeoDataFrame, crs="EPSG:4326")
)

In [4]:
caracas_coverage.explore(
    style_kwds=dict(
        marker_kwds=dict(radius=3),
        style_kwds=dict(fillOpacity=0.6, weight=0),
    ),
    tiles=BASE_TILES,
)

In [5]:
caracas_coverage.explore(
    "days_present",
    cmap="Greens",
    style_kwds=dict(
        marker_kwds=dict(radius=3, fill=True),
        style_kwds=dict(fillOpacity=0.6, weight=0),
    ),
    tiles=BASE_TILES,
)

In [6]:
national_coverage.explore(
    cmap="Greens",
    style_kwds=dict(
        marker_kwds=dict(radius=3, fill=True),
        style_kwds=dict(fillOpacity=0.6, weight=0),
    ),
    tiles=BASE_TILES,
)

In [7]:
national_coverage.explore(
    "days_present",
    cmap="Greens",
    style_kwds=dict(
        marker_kwds=dict(radius=3, fill=True),
        style_kwds=dict(fillOpacity=0.6, weight=0),
    ),
    tiles=BASE_TILES,
)